# Reversibility project: full run on a Colab GPU
Runtime > Change runtime type > T4 GPU. Run the cells one at a time, top to bottom.
Inputs (sandbox, supplement, ScienceWorld cut-off, reworded test copies) are frozen files inside the zip; nothing is regenerated.
If an older run is still in `/content/reversibility`, download its `results/` first: it is your pilot evidence.

In [ ]:
%cd /content
from google.colab import files
files.upload()  # choose reversibility-3.zip

In [ ]:
!unzip -oq /content/reversibility-3.zip -d /content/run3
%cd /content/run3/reversibility
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi | head -15

In [ ]:
# STEP 1: quick plumbing test (~10-15 min). Numbers are meaningless; it only checks that nothing crashes.
!QUICK=1 MODEL=Qwen/Qwen2.5-0.5B-Instruct TAG=quick bash run_all.sh 2>&1 | tail -40
# throw the plumbing-test outputs away so they cannot end up in the real tables and figures
!rm -rf results/*quick* figures/*quick* data/h_quick_* data/quick

In [ ]:
# STEP 2: agent pilot (6 task pairs, other instances than the eval set, ~5 min). Does the shortcut tempt the model?
!PILOT=1 MODEL=Qwen/Qwen2.5-3B-Instruct TAG=qwen3b bash run_all.sh > results/pilot_qwen3b.log 2>&1; tail -3 results/pilot_qwen3b.log
import json
d = json.load(open('results/agent_qwen3b_pilot.json'))
print({k: d[k] for k in ('target_action_rate_irr_version', 'target_action_rate_rev_version', 'caution_rate_irr', 'over_caution_rate_rev', 'K_states_correct_version', 'parse_failures_total')})
for t, v in d['per_template'].items(): print(t, v)
# garbage replies / parse failures on most steps = fp16 trouble: rerun this cell with BF16=1 in front of PILOT=1

In [ ]:
# STEP 3: the real run (~3 h on a T4; keep the tab open). Check the pilot output first.
!MODEL=Qwen/Qwen2.5-3B-Instruct TAG=qwen3b bash run_all.sh 2>&1 | tee results/run_qwen3b.log | grep -v '%|'

In [ ]:
# save right away (Colab can disconnect): results, figures, every prompt
!zip -rq results_qwen3b.zip results figures data/h_qwen3b_*_prompts.jsonl && echo ok
files.download('results_qwen3b.zip')

In [ ]:
# BONUS RQ4 steering (~2-3 h), only if time is left. Layer = binary best layer chosen on validation.
import json
L = json.load(open('results/probe_qwen3b_last.json'))['tasks']['binary']['best_layer']; print('layer', L)
!python steering.py --model Qwen/Qwen2.5-3B-Instruct --data data/sandbox.jsonl data/sandbox_extra.jsonl --hidden data/h_qwen3b_sandbox data/h_qwen3b_sandbox_extra --layer {L} --tag qwen3b && python make_tables.py --model-tag qwen3b

In [ ]:
# LLAMA (second model family), only after Meta approves access. Needs the HF_TOKEN secret (key icon, left sidebar).
# 4-bit NF4 so the 8B model fits the T4: a deviation to state in the paper.
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!pip install -q bitsandbytes
!LOAD_4BIT=1 MODEL=meta-llama/Llama-3.1-8B-Instruct TAG=llama8b bash run_all.sh 2>&1 | tee results/run_llama8b.log | grep -v '%|'
!zip -rq results_llama8b.zip results figures data/h_llama8b_*_prompts.jsonl && echo ok
files.download('results_llama8b.zip')